# 🔌 The API Surface — five scenarios that decide your error design

**Birlasoft FORGE FDE Academy · Sprint 1 Day 17 · DecisionStream AI**

---

The pipeline works. From today somebody else's code calls it — which means
every way they can get it wrong is now **your design decision**, made either
deliberately or by default.

> An API is a promise about what happens when things go wrong.
> The happy path is the easy half.

### Five scenarios

The full lab has twenty. These five carry the argument.

| | Scenario | The question |
|---|---|---|
| **1** | Client sends an unexpected field | **Where** should we reject it, and why? |
| **2** | The document is unreadable | 422 or 500? |
| **3** | The model returns invalid JSON | 422 or 500? |
| **4** | Client times out and retries | How do we prevent duplicate processing? |
| **5** | Content filter fails **mid-stream** | Can we still return 422? |

Scenario 5 is the one that reshapes your architecture, and it is demonstrated
here against a **real streaming response** rather than described.

### And one concept that outranks all five

Most people stop at *504 means retry, 422 means don't*.

> **An HTTP status is a category. `retryable` is an instruction.**

```json
{ "status": 504, "retryable": true, "retry_after_seconds": 60 }
```

The FDE question is not whether you emit that field. It is:

**Does the consuming team's client actually branch on it?**

If nobody consumes it, you have designed *documentation*, not *behaviour*.
Section 8 turns that from a talking point into a number.

---

### What is real here

| Part | Real or simulated |
|---|---|
| Scenario 3 — invalid model JSON | **Real** — forced by truncation against your endpoint |
| Scenario 5 — mid-stream failure | **Real** — a live stream, interrupted |
| Scenarios 1, 2, 4 | **Local** — gateway and handler code you can read and edit |
| The retryable experiment | **Arithmetic** over real error taxonomies |

### Before you run anything

Your key is a credential. Masked prompt, never printed. A shared Colab notebook
carries its outputs.

Cost is small — roughly **12 calls** for the whole notebook.

## 1 · Install and import

In [1]:
%pip install -q openai jsonschema python-dotenv

import os, re, json, time, uuid, getpass, warnings
from dataclasses import dataclass, field as dc_field
from typing import Any

from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings("ignore")
print("imports ready")

Note: you may need to restart the kernel to use updated packages.
imports ready


## 2 · Your Azure credentials

Endpoint, key, and the **deployment name** (the name you gave the model in your
resource, not the model name).

Leave blank to run simulated — scenarios 1, 2 and 4 are unaffected, and 3 and 5
fall back to a scripted equivalent.

In [2]:
AOAI_ENDPOINT   = os.environ.get("AOAI_ENDPOINT", "").strip()
AOAI_KEY        = os.environ.get("AOAI_KEY", "").strip()
AOAI_DEPLOYMENT = os.environ.get("AOAI_DEPLOYMENT", "").strip()
API_VERSION     = "2024-10-21"    # VERIFY against current docs before relying on it

if not AOAI_ENDPOINT:
    AOAI_ENDPOINT = input("Azure OpenAI / Foundry endpoint (blank = simulated): ").strip()
if AOAI_ENDPOINT:
    if not AOAI_KEY:
        AOAI_KEY = getpass.getpass("API key (hidden): ").strip()
    if not AOAI_DEPLOYMENT:
        AOAI_DEPLOYMENT = input("Deployment name: ").strip()

LIVE = bool(AOAI_ENDPOINT and AOAI_KEY and AOAI_DEPLOYMENT)
client = None
if LIVE:
    from openai import AzureOpenAI
    client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_KEY,
                         api_version=API_VERSION, max_retries=0, timeout=60.0)
    print(f"mode:       LIVE\ndeployment: {AOAI_DEPLOYMENT}")
else:
    print("mode: SIMULATED — nothing billed.")

mode:       LIVE
deployment: gpt-4.1-mini


## 3 · The contract

Two things every error in this API carries.

**RFC 9457 problem details** — a standard JSON error shape, so every client
parses errors the same way instead of each one inventing a parser.

**`retryable`** — the field that earns its place. A status code tells the
caller roughly what happened. `retryable` tells them **what to do**.

In [3]:
def problem(status, title, detail, *, retryable, retry_after=None, **extra):
    """
    RFC 9457 problem details.

    `retryable` is not decoration. A client library branches on it, and a
    caller that cannot tell "try again in a minute" from "never try this
    again" will either retry forever or give up on something that would
    have worked.
    """
    b = {"type": f"https://errors.decisionstream.example/{title}",
         "title": title, "status": status, "detail": detail,
         "retryable": retryable}
    if retry_after is not None:
        b["retry_after_seconds"] = retry_after
    b.update(extra)
    return b


SCHEMA = {
    "required": ["case_reference", "document_refs", "decision_date"],
    "types": {"case_reference": str, "document_refs": list,
              "decision_date": str, "callback_url": str},
    "patterns": {"case_reference": r"^CLM-\d{4}-\d{4}$",
                 "decision_date": r"^\d{4}-\d{2}-\d{2}$"},
    "additional_properties": False,     # <- Scenario 1 lives here
    "max_document_refs": 20,
}

# What a valid decision profile must look like coming back from the model.
PROFILE_SCHEMA = {
    "type": "object",
    "required": ["case_reference", "extracted_fields", "policy_basis",
                 "needs_human"],
    "properties": {
        "case_reference":   {"type": "string"},
        "extracted_fields": {"type": "object"},
        "policy_basis":     {"type": "array"},
        "needs_human":      {"type": "boolean"},
    },
    "additionalProperties": True,
}

print(json.dumps(problem(504, "extraction-timeout",
                         "Extraction did not complete in time.",
                         retryable=True, retry_after=60), indent=2))

{
  "type": "https://errors.decisionstream.example/extraction-timeout",
  "title": "extraction-timeout",
  "status": 504,
  "detail": "Extraction did not complete in time.",
  "retryable": true,
  "retry_after_seconds": 60
}


## 4 · The two layers

`gateway()` is API Management. `handler()` is the Azure Function.

Read the split. **Anything decidable without the case data belongs at the
gateway** — it costs milliseconds and nothing. Everything past it costs a
Document Intelligence call and a model call.

In [4]:
COST = {"gateway_ms": 2, "gateway_usd": 0.0,
        "doc_intel_ms": 9000, "doc_intel_usd": 0.010,
        "generation_ms": 2600, "generation_usd": 0.004}

@dataclass
class Response:
    status: int
    body: dict
    ms: float = 0.0
    usd: float = 0.0
    rejected_at: str = "-"
    @property
    def retryable(self): return self.body.get("retryable")

SEEN_KEYS: dict[str, dict] = {}      # idempotency store — Scenario 4

def gateway(req, *, validate_schema=True):
    """API Management. Returns a Response to reject, or None to pass through."""
    ms = COST["gateway_ms"]

    if not req.get("_token"):
        return Response(401, problem(401, "unauthenticated",
                        "No credential presented.", retryable=False), ms, 0.0, "gateway")

    key = req.get("_idempotency_key")
    if not key:
        return Response(400, problem(400, "missing-idempotency-key",
                        "Idempotency-Key header is required.", retryable=False),
                        ms, 0.0, "gateway")
    if key in SEEN_KEYS:
        prior = SEEN_KEYS[key]
        if prior["fingerprint"] == json.dumps(req.get("body"), sort_keys=True):
            return Response(202, {**prior["response"],
                                  "note": "duplicate request — original returned"},
                            ms, 0.0, "gateway")
        return Response(409, problem(409, "idempotency-key-reused",
                        "This key was used with different content.", retryable=False),
                        ms, 0.0, "gateway")

    if not validate_schema:
        return None

    body = req.get("body")
    if not isinstance(body, dict):
        return Response(400, problem(400, "malformed-json",
                        "Body is not a JSON object.", retryable=False), ms, 0.0, "gateway")

    missing = [f for f in SCHEMA["required"] if f not in body]
    if missing:
        return Response(400, problem(400, "schema-violation",
                        f"Missing required field(s): {', '.join(missing)}",
                        retryable=False, fields=missing), ms, 0.0, "gateway")

    if not SCHEMA["additional_properties"]:
        extra = [k for k in body if k not in SCHEMA["types"]]
        if extra:
            return Response(400, problem(400, "schema-violation",
                            f"Unexpected field(s): {', '.join(extra)}",
                            retryable=False, fields=extra), ms, 0.0, "gateway")

    for f, t in SCHEMA["types"].items():
        if f in body and not isinstance(body[f], t):
            return Response(400, problem(400, "schema-violation",
                            f"Field '{f}' has the wrong type.",
                            retryable=False, fields=[f]), ms, 0.0, "gateway")

    for f, pat in SCHEMA["patterns"].items():
        if f in body and not re.match(pat, str(body[f])):
            return Response(400, problem(400, "schema-violation",
                            f"Field '{f}' does not match the required format.",
                            retryable=False, fields=[f]), ms, 0.0, "gateway")
    return None


def handler(req, *, gateway_validated=True):
    """The Azure Function. Where the money gets spent."""
    ms, usd = 120.0, 0.0

    if not gateway_validated:
        body = req.get("body")
        if not isinstance(body, dict) or [f for f in SCHEMA["required"] if f not in body]:
            return Response(400, problem(400, "schema-violation",
                            "Request failed validation.", retryable=False),
                            ms, usd, "function")

    ms += COST["doc_intel_ms"]; usd += COST["doc_intel_usd"]
    if req.get("_doc_unreadable"):                      # Scenario 2
        return Response(422, problem(422, "document-unreadable",
                        "The document could not be read. Supply a clearer scan "
                        "or a text-based PDF.", retryable=False), ms, usd, "upstream")
    if req.get("_doc_timeout"):
        return Response(504, problem(504, "extraction-timeout",
                        "Extraction did not complete in time.",
                        retryable=True, retry_after=60), ms, usd, "upstream")

    ms += COST["generation_ms"]; usd += COST["generation_usd"]
    if req.get("_bad_model_json"):                      # Scenario 3
        return Response(500, problem(500, "invalid-model-output",
                        "The model response failed schema validation.",
                        retryable=True, retry_after=5), ms, usd, "upstream")

    return Response(202, {"assessment_id": f"as-{uuid.uuid4().hex[:8]}",
                          "status": "pending"}, ms, usd, "-")


def call_api(req, *, validate_at_gateway=True):
    r = gateway(req, validate_schema=validate_at_gateway)
    if r is not None:
        return r
    fr = handler(req, gateway_validated=validate_at_gateway)
    fr.ms += COST["gateway_ms"]
    if fr.status < 300 and req.get("_idempotency_key"):
        SEEN_KEYS[req["_idempotency_key"]] = {
            "fingerprint": json.dumps(req.get("body"), sort_keys=True),
            "response": fr.body}
    return fr


def valid_body(**over):
    b = {"case_reference": "CLM-2026-4471",
         "document_refs": ["blob://pkg/case_form.pdf"],
         "decision_date": "2026-03-14"}
    b.update(over); return b

def make_req(**over):
    r = {"_token": "eyJ...", "_idempotency_key": f"key-{uuid.uuid4().hex[:8]}",
         "body": valid_body()}
    r.update(over); return r

def show(label, r):
    print(f"{label}\n  status {r.status}   retryable={r.retryable}   "
          f"rejected at {r.rejected_at}   {r.ms:,.0f}ms   ${r.usd:.4f}")
    print("  " + json.dumps(r.body, indent=2)[:400].replace("\n", "\n  ") + "\n")

print("gateway and handler loaded")

gateway and handler loaded


---

# Scenario 1 · The unexpected field

**The question is not whether to reject it. It is WHERE.**

A consuming team sends `applicant_ssn` — a field nobody asked for, which their
system happens to have to hand.

Run both designs and compare what it costs to reach the same conclusion.

In [5]:
req = make_req(body=valid_body(applicant_ssn="123-45-6789"))

a = call_api(dict(req), validate_at_gateway=True)
show("VALIDATED AT THE GATEWAY", a)

req2 = dict(req); req2["_idempotency_key"] = f"key-{uuid.uuid4().hex[:8]}"
b = call_api(req2, validate_at_gateway=False)
show("VALIDATED INSIDE THE FUNCTION", b)

print(f"""Same conclusion, different price:

  gateway   {a.status}   {a.ms:>7,.0f} ms   ${a.usd:.4f}
  function  {b.status}   {b.ms:>7,.0f} ms   ${b.usd:.4f}

But the cost is not the headline. Look at what the Function returned.

Hand-written validation inside a handler is always THINNER than a declarative
schema at the edge, because somebody writes the required-field check on day one
and nobody goes back to add the pattern, type and bound checks.

So `applicant_ssn` was accepted and processed. Nobody asked for it, there is no
lawful basis to hold it, and it is now in your logs, your storage and your
subject access response.

Rejecting unknown fields is a DATA PROTECTION control that happens to also be
a cost control.
""")

VALIDATED AT THE GATEWAY
  status 400   retryable=False   rejected at gateway   2ms   $0.0000
  {
    "type": "https://errors.decisionstream.example/schema-violation",
    "title": "schema-violation",
    "status": 400,
    "detail": "Unexpected field(s): applicant_ssn",
    "retryable": false,
    "fields": [
      "applicant_ssn"
    ]
  }

VALIDATED INSIDE THE FUNCTION
  status 202   retryable=None   rejected at -   11,722ms   $0.0140
  {
    "assessment_id": "as-a9e440cf",
    "status": "pending"
  }

Same conclusion, different price:

  gateway   400         2 ms   $0.0000
  function  202    11,722 ms   $0.0140

But the cost is not the headline. Look at what the Function returned.

Hand-written validation inside a handler is always THINNER than a declarative
schema at the edge, because somebody writes the required-field check on day one
and nobody goes back to add the pattern, type and bound checks.

So `applicant_ssn` was accepted and processed. Nobody asked for it, there is no
l

### Discuss — 5 minutes

The consuming team asks you to relax the schema, because changing their code is
inconvenient.

1. Accepting unknown fields costs nothing today. What does it cost the first
   time somebody sends a field you have no lawful basis to hold?
2. If you accept it — is it in your logs? Your storage? Your subject access
   response? Who decided that?
3. **Is there a version of yes**, and what would it require them to tell you
   first?

---

# Scenario 2 · The unreadable document

**422 or 500?**

Decide before you run the cell.

In [6]:
r = call_api(make_req(_doc_unreadable=True))
show("unreadable document", r)

print("""422 — because the CALLER can fix it.

The test that resolves nearly every one of these:

    Can the caller do something differently and succeed?
      yes -> 4xx
      no  -> 5xx

A 500 here tells them it is our fault and to try again, and it will fail
identically every time. The detail line matters as much as the code: "supply a
clearer scan or a text-based PDF" is an instruction. "Internal server error"
is a support ticket.

Note retryable=False. The status says what happened; retryable says what to do.
""")

unreadable document
  status 422   retryable=False   rejected at upstream   9,122ms   $0.0100
  {
    "type": "https://errors.decisionstream.example/document-unreadable",
    "title": "document-unreadable",
    "status": 422,
    "detail": "The document could not be read. Supply a clearer scan or a text-based PDF.",
    "retryable": false
  }

422 — because the CALLER can fix it.

The test that resolves nearly every one of these:

    Can the caller do something differently and succeed?
      yes -> 4xx
      no  -> 5xx

A 500 here tells them it is our fault and to try again, and it will fail
identically every time. The detail line matters as much as the code: "supply a
clearer scan or a text-based PDF" is an instruction. "Internal server error"
is a support ticket.

Note retryable=False. The status says what happened; retryable says what to do.



---

# Scenario 3 · The model returns invalid JSON

**422 or 500?**

This one is demonstrated for real. We ask your endpoint for a strict JSON
decision profile, then **truncate it** with a low `max_tokens` — which is one
of the most common ways real model output fails schema validation.

Truncated JSON is not valid JSON.

In [7]:
import jsonschema

PROFILE_PROMPT = """Return ONLY a JSON object for this claim, no prose, no markdown fences:
{"case_reference": str, "extracted_fields": {...}, "policy_basis": [...], "needs_human": bool}

Claim CLM-2026-4471. Front nearside collision 14 March 2026 (page 1).
Repair estimate GBP 12,400.00 (page 7). Policy limit GBP 25,000.00 (page 3).
Excess GBP 350.00 (page 3). Supplementary estimate GBP 14,750.00 (page 11).
Include every figure with its page in extracted_fields, and cite the clauses
you relied on in policy_basis."""

def ask_for_profile(max_tokens):
    if not LIVE:
        full = json.dumps({"case_reference": "CLM-2026-4471",
                           "extracted_fields": {"repair_estimate_total": 12400.0},
                           "policy_basis": [{"clause": "4.1"}], "needs_human": False})
        return full if max_tokens >= 200 else full[:int(len(full) * 0.55)]
    r = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[{"role": "system", "content": "You output only valid JSON."},
                  {"role": "user", "content": PROFILE_PROMPT}],
        max_tokens=max_tokens, temperature=0.0)
    return r.choices[0].message.content or ""

def validate_profile(text):
    """Returns (ok, parsed_or_None, why)."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        obj = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return False, None, f"not parseable as JSON: {e}"
    try:
        jsonschema.validate(obj, PROFILE_SCHEMA)
    except jsonschema.ValidationError as e:
        return False, obj, f"parsed but failed schema: {e.message[:90]}"
    return True, obj, "valid"

for label, mt in (("generous max_tokens", 400), ("TRUNCATED max_tokens", 60)):
    text = ask_for_profile(mt)
    ok, obj, why = validate_profile(text)
    print(f"{label} ({mt})")
    print(f"  returned {len(text)} chars   valid: {ok}")
    print(f"  {why}")
    print(f"  ...{text[-90:].strip()}\n")

    if not ok:
        r = Response(500, problem(500, "invalid-model-output",
                     "The model response failed schema validation.",
                     retryable=True, retry_after=5), 11720, 0.014, "upstream")
        show("  -> what the API returns", r)

print("""500 — because the CALLER did nothing to cause this.

Returning a 4xx here blames them for our problem. They sent a well-formed
request; our model produced output that failed our own schema.

retryable=True, because a re-run may well succeed. And this one should PAGE
somebody — a rising rate of invalid model output is a real incident, not a
client integration issue.

Notice what caught it: a schema check on the model's output, not a status code
from the provider. The API call SUCCEEDED. Every layer reported healthy. Only
validating the content found the problem.
""")

generous max_tokens (400)
  returned 807 chars   valid: True
  valid
  ..."Clause 7.3 - Repair estimates and supplementary claims"
  ],
  "needs_human": false
}

TRUNCATED max_tokens (60)
  returned 188 chars   valid: False
  not parseable as JSON: Expecting ',' delimiter: line 7 column 22 (char 188)
  ...de",
    "collision_date": "14 March 2026",
    "repair_estimate": {
      "amount": 12400

  -> what the API returns
  status 500   retryable=True   rejected at upstream   11,720ms   $0.0140
  {
    "type": "https://errors.decisionstream.example/invalid-model-output",
    "title": "invalid-model-output",
    "status": 500,
    "detail": "The model response failed schema validation.",
    "retryable": true,
    "retry_after_seconds": 5
  }

500 — because the CALLER did nothing to cause this.

Returning a 4xx here blames them for our problem. They sent a well-formed
request; our model produced output that failed our own schema.

retryable=True, because a re-run may well succeed. And th

---

# Scenario 4 · The client times out and retries

The client sends `POST /case-application`, the connection drops before the
response arrives, and it retries — which is **exactly what you want it to do**.

How do you stop that creating two assessments?

In [8]:
SEEN_KEYS.clear()
KEY = "idem-7f3a91"

first = call_api(make_req(_idempotency_key=KEY))
show("1. first attempt (response lost in transit)", first)

replay = call_api(make_req(_idempotency_key=KEY))
show("2. client retries with the SAME key, same body", replay)

conflict = call_api(make_req(_idempotency_key=KEY,
                             body=valid_body(case_reference="CLM-2026-9999")))
show("3. same key, DIFFERENT body", conflict)

nokey = call_api(make_req(_idempotency_key=None))
show("4. no key at all", nokey)

print(f"""The distinction that gets implemented wrongly:

  same key, same body       202  return the ORIGINAL acceptance
  same key, different body  409  a genuine conflict — their bug, they need to know
  no key                    400  reject it

assessment_id first attempt: {first.body.get('assessment_id')}
assessment_id on replay:     {replay.body.get('assessment_id')}
                             ^ the same. One assessment, not two.

THE MISTAKE TO AVOID: returning 409 for a straightforward replay.

The caller did nothing wrong. Their connection dropped and they retried, which
is the behaviour you want. Punishing correct client behaviour with an error
trains them to stop retrying — and then transient failures become lost cases.
""")

1. first attempt (response lost in transit)
  status 202   retryable=None   rejected at -   11,722ms   $0.0140
  {
    "assessment_id": "as-dcca315a",
    "status": "pending"
  }

2. client retries with the SAME key, same body
  status 202   retryable=None   rejected at gateway   2ms   $0.0000
  {
    "assessment_id": "as-dcca315a",
    "status": "pending",
    "note": "duplicate request \u2014 original returned"
  }

3. same key, DIFFERENT body
  status 409   retryable=False   rejected at gateway   2ms   $0.0000
  {
    "type": "https://errors.decisionstream.example/idempotency-key-reused",
    "title": "idempotency-key-reused",
    "status": 409,
    "detail": "This key was used with different content.",
    "retryable": false
  }

4. no key at all
  status 400   retryable=False   rejected at gateway   2ms   $0.0000
  {
    "type": "https://errors.decisionstream.example/missing-idempotency-key",
    "title": "missing-idempotency-key",
    "status": 400,
    "detail": "Idempotency-Key

---

# Scenario 5 · The content filter fails mid-stream

**Can we still return 422?**

This runs a **real** streaming response and interrupts it partway through.
Watch the timeline.

In [9]:
FAIL_AT_CHUNK = 12

def stream_with_failure(fail_at=FAIL_AT_CHUNK):
    events, sent = [], ""
    t0 = time.perf_counter()
    def stamp(): return (time.perf_counter() - t0) * 1000

    events.append((0.0, "http_status", "200 OK — COMMITTED. No going back."))

    if not LIVE:
        chunks = ["The claim relates to a front nearside collision ",
                  "on 14 March 2026 (page 1). ", "The repair estimate totals ",
                  "GBP 12,400.00 (page 7) ", "against a policy limit of ",
                  "GBP 25,000.00 (page 3). ", "A supplementary estimate at ",
                  "page 11 raises the total to ", "GBP 14,750.00, an increase ",
                  "of 18.9%. ", "This is below the twenty ", "per cent threshold ",
                  "in clause 3.5. ", "No further action is indicated."]
        for i, c in enumerate(chunks):
            time.sleep(0.06)
            if i == fail_at:
                events.append((stamp(), "error",
                               "content filter blocked the remainder"))
                return events, sent, False
            sent += c
            events.append((stamp(), "chunk", f"{len(c)} chars"))
        events.append((stamp(), "done", f"{len(sent)} chars"))
        return events, sent, True

    stream = client.chat.completions.create(
        model=AOAI_DEPLOYMENT,
        messages=[{"role": "system", "content": "You summarise motor claims for a handler."},
                  {"role": "user", "content":
                   "Summarise: claim CLM-2026-4471, front nearside collision 14 March "
                   "2026 (page 1), repair estimate GBP 12,400.00 (page 7), policy limit "
                   "GBP 25,000.00 (page 3), supplementary GBP 14,750.00 (page 11). "
                   "Write about 120 words, citing pages."}],
        max_tokens=300, temperature=0.0, stream=True)

    n = 0
    for chunk in stream:
        if not chunk.choices:
            continue
        d = chunk.choices[0].delta
        if not getattr(d, "content", None):
            continue
        if n == fail_at:
            events.append((stamp(), "error", "content filter blocked the remainder"))
            stream.close()
            return events, sent, False
        sent += d.content
        n += 1
        events.append((stamp(), "chunk", f"{len(d.content)} chars"))
    events.append((stamp(), "done", f"{len(sent)} chars"))
    return events, sent, True

events, sent, completed = stream_with_failure()

print(f"{'t':>9}   event")
print("-" * 62)
for t, kind, note in events:
    if kind in ("http_status", "error", "done"):
        print(f"{t:>8,.0f}ms   {kind.upper():<12} {note}")
print(f"\ncompleted: {completed}")
print(f"characters already on the reader's screen: {len(sent)}\n")
print("what the handler is looking at right now:")
print("  " + (sent[-200:] if sent else "(nothing)") + " ...")

print(f"""
NO. You cannot return 422.

You sent 200 OK at time zero — {events[-1][0]:,.0f} ms before this failed. The
status code was decided before you knew whether the work would succeed.

Your options, and there is no third one:
  - emit an SSE error event and close   (only helps if the client handles it,
                                         and most do not, because the happy
                                         path works and nobody tested this)
  - close the connection abruptly       (a truncated answer that LOOKS complete)

The reader has half an assessment on screen and no indication that it is half.
That is worse than an error page, because they may act on it.
""")

        t   event
--------------------------------------------------------------
       0ms   HTTP_STATUS  200 OK — COMMITTED. No going back.
   1,083ms   ERROR        content filter blocked the remainder

completed: False
characters already on the reader's screen: 36

what the handler is looking at right now:
  Claim CLM-2026-4471 involves a front ...

NO. You cannot return 422.

You sent 200 OK at time zero — 1,083 ms before this failed. The
status code was decided before you knew whether the work would succeed.

Your options, and there is no third one:
  - emit an SSE error event and close   (only helps if the client handles it,
                                         and most do not, because the happy
                                         path works and nobody tested this)
  - close the connection abruptly       (a truncated answer that LOOKS complete)

The reader has half an assessment on screen and no indication that it is half.
That is worse than an error page, because they 

### The question that reshapes the architecture

**What should we have validated before the first byte?**

Everything. Auth, schema, case state, retrieval success, content policy on the
prompt — all of it has to succeed **before** you commit to a status code.

> Streaming does not remove error handling.
> It moves all of it earlier, and makes the part that happens later much harder.

And the rule that follows from this one scenario:

> ### Never stream the structured decision profile.
>
> A partially streamed JSON object is not valid JSON. You cannot schema-validate
> it, you cannot check for forbidden fields, and you cannot refuse to release it
> — because you already sent 200 OK.
>
> **Stream prose to a human. Return structure as a whole, validated.**

Which resolves the clash in your brief: `POST /case-application` returns 202
with a ticket. `GET /case-assessments/{id}/narrative` streams.

---

# 8 · `retryable` — documentation, or behaviour?

Most people stop at *504 means retry, 422 means don't*. That is a category.

> **An HTTP status is a category. `retryable` is an instruction.**

The FDE question is not whether you emit the field. It is:

### Does the consuming team's client actually branch on it?

If nobody consumes it, you have designed documentation.

This section makes that concrete. Three client implementations, the same
thousand requests, and a count of what each one wastes.

In [10]:
import random

# A realistic error mix over 1,000 requests.
#
# TWO ROWS MATTER MORE THAN THE REST, because they are where the status code
# and the instruction DISAGREE:
#
#   rate-limited        429 is a 4xx and IS retryable
#   unsupported-format  500 is a 5xx and is NOT retryable — it will fail
#                       identically every single time
#
# A client using "retry 5xx" gets both of those backwards. That is not a
# contrived example; 429 is one of the most common errors any API returns.
ERROR_MIX = [
    ("success",              0.70, 202, None,  None),
    ("schema-violation",     0.05, 400, False, None),
    ("document-unreadable",  0.04, 422, False, None),
    ("content-filtered",     0.03, 422, False, None),
    ("rate-limited",         0.05, 429, True,  30),   # retryable 4xx
    ("unsupported-format",   0.03, 500, False, None), # NON-retryable 5xx
    ("extraction-timeout",   0.04, 504, True,  60),
    ("model-capacity",       0.03, 503, True,  15),
    ("invalid-model-output", 0.03, 500, True,  5),
]

N_REQUESTS = 1000
MAX_ATTEMPTS = 4
rng = random.Random(17)

def draw():
    r, acc = rng.random(), 0.0
    for name, p, status, retryable, ra in ERROR_MIX:
        acc += p
        if r <= acc:
            return name, status, retryable, ra
    return ERROR_MIX[-1][0], 500, True, 5

def client_naive(status, retryable):
    """Retries anything that is not a 2xx. Very common."""
    return status >= 300

def client_status_only(status, retryable):
    """Branches on the status code alone — the '5xx means retry' rule."""
    return status >= 500

def client_reads_retryable(status, retryable):
    """Branches on the field we designed."""
    return bool(retryable)

CLIENTS = {"naive — retry anything non-2xx": client_naive,
           "status code only — retry 5xx":   client_status_only,
           "reads `retryable`":              client_reads_retryable}

print(f"{'client':<34}{'calls':>8}{'wasted':>9}{'lost cases':>12}{'$ wasted':>11}")
print("-" * 76)
results = {}
for label, fn in CLIENTS.items():
    calls = wasted = lost = 0
    rng.seed(17)
    for _ in range(N_REQUESTS):
        name, status, retryable, ra = draw()
        calls += 1
        if status < 300:
            continue
        attempts = 1
        while fn(status, retryable) and attempts < MAX_ATTEMPTS:
            attempts += 1; calls += 1
            if retryable:
                # a genuinely transient failure usually clears
                if rng.random() < 0.6:
                    status, retryable = 202, None
                    break
            else:
                wasted += 1        # will never succeed, and we paid for it
        # a transient failure the client refused to retry is a lost case
        if status >= 300 and retryable and not fn(status, retryable):
            lost += 1
    cost = wasted * (COST["doc_intel_usd"] + COST["generation_usd"])
    results[label] = (calls, wasted, lost, cost)
    print(f"{label:<34}{calls:>8,}{wasted:>9,}{lost:>12,}{cost:>10,.2f}")

print(f"""
Read the two failure columns.

WASTED calls are retries of something that will NEVER succeed — a 422 for an
unreadable document retried four times is four Document Intelligence calls to
reach the same answer. The naive client does this constantly.

LOST CASES are the opposite failure: a transient blip the client refused to
retry, so a case sits unprocessed and nobody finds out until somebody chases it.

Now look at the middle row. The status-code-only client is the one that looks
professional, and it fails in BOTH directions at once — because two of our
error types have a status code that disagrees with the instruction:

  rate-limited        429  retryable=True   a 4xx it SHOULD retry and does not
  unsupported-format  500  retryable=False  a 5xx it retries and never should

Neither is contrived. A 429 is one of the most common errors any API returns,
and a deterministic 500 is what a poison message looks like.

"5xx means retry" is a heuristic. It is not wrong often, and when it is wrong
it is wrong quietly and in both directions.

Only the client that reads `retryable` gets both columns to zero — and it can
only do that because somebody emitted the field AND somebody else read it.
""")

client                               calls   wasted  lost cases   $ wasted
----------------------------------------------------------------------------
naive — retry anything non-2xx       1,697      474           0      6.64
status code only — retry 5xx         1,246       78          35      1.09
reads `retryable`                    1,223        0           0      0.00

Read the two failure columns.

WASTED calls are retries of something that will NEVER succeed — a 422 for an
unreadable document retried four times is four Document Intelligence calls to
reach the same answer. The naive client does this constantly.

LOST CASES are the opposite failure: a transient blip the client refused to
retry, so a case sits unprocessed and nobody finds out until somebody chases it.

Now look at the middle row. The status-code-only client is the one that looks
professional, and it fails in BOTH directions at once — because two of our
error types have a status code that disagrees with the instructio

### The question to take to the consuming team

You have now designed a field that eliminates two entire classes of failure —
**if somebody reads it.**

Ask them, in writing, before go-live:

> *"Our error bodies carry a `retryable` boolean and a `retry_after_seconds`.
> Does your client branch on those, or on the status code? If the status code —
> here is why that breaks on two of our error types."*

If the answer is "we retry 5xx", you have three options and they are all fine
as long as you **choose** one:

1. Change your taxonomy so the heuristic works — no retryable 5xx
2. Get them to read the field
3. Accept the waste, and know the number

What is not fine is emitting the field, documenting it, and assuming.

> **If nobody consumes it, you have designed documentation, not behaviour.**

---

## 9 · Export and the ADR

In [11]:
taxonomy = [
    {"situation": "unexpected field in request",  "status": 400, "retryable": False,
     "reject_at": "gateway", "why": "caller can remove it; also a data protection control"},
    {"situation": "missing idempotency key",      "status": 400, "retryable": False,
     "reject_at": "gateway", "why": "without it one timeout creates two assessments"},
    {"situation": "replayed key, same body",      "status": 202, "retryable": None,
     "reject_at": "gateway", "why": "return the original — correct client behaviour"},
    {"situation": "replayed key, different body", "status": 409, "retryable": False,
     "reject_at": "gateway", "why": "a genuine conflict; their bug"},
    {"situation": "document unreadable",          "status": 422, "retryable": False,
     "reject_at": "upstream", "why": "caller can supply a clearer scan"},
    {"situation": "extraction timeout",           "status": 504, "retryable": True,
     "reject_at": "upstream", "why": "ours, transient — give a Retry-After"},
    {"situation": "content filter blocked",       "status": 422, "retryable": False,
     "reject_at": "upstream", "why": "will not succeed on retry; name the category"},
    {"situation": "model at capacity",            "status": 503, "retryable": True,
     "reject_at": "upstream", "why": "not 429 — that is our quota on them"},
    {"situation": "model returned invalid JSON",  "status": 500, "retryable": True,
     "reject_at": "upstream", "why": "our problem, not the caller's; should page"},
    {"situation": "no policy retrieved",          "status": 200, "retryable": None,
     "reject_at": "-", "why": "not an error — a refusal. needs_human=true"},
]

print(f"{'situation':<32}{'code':>6}{'retry':>8}  reject at   why")
print("-" * 108)
for t in taxonomy:
    print(f"{t['situation']:<32}{t['status']:>6}{str(t['retryable']):>8}  "
          f"{t['reject_at']:<11} {t['why']}")

out = {"taxonomy": taxonomy,
       "streaming": {
           "POST /case-application": "202 + ticket, async, never streams",
           "GET /case-assessments/{id}/narrative": "streams prose to a human",
           "rule": "never stream the structured decision profile"},
       "retryable_clients": {k: {"calls": v[0], "wasted": v[1], "lost": v[2],
                                 "usd_wasted": round(v[3], 2)}
                             for k, v in results.items()},
       "mode": "live" if LIVE else "simulated"}
Path = __import__("pathlib").Path
Path("api_taxonomy.json").write_text(json.dumps(out, indent=2))
print("\nwritten: api_taxonomy.json")

try:
    from google.colab import files
    files.download("api_taxonomy.json")
except Exception:
    pass

situation                         code   retry  reject at   why
------------------------------------------------------------------------------------------------------------
unexpected field in request        400   False  gateway     caller can remove it; also a data protection control
missing idempotency key            400   False  gateway     without it one timeout creates two assessments
replayed key, same body            202    None  gateway     return the original — correct client behaviour
replayed key, different body       409   False  gateway     a genuine conflict; their bug
document unreadable                422   False  upstream    caller can supply a clearer scan
extraction timeout                 504    True  upstream    ours, transient — give a Retry-After
content filter blocked             422   False  upstream    will not succeed on retry; name the category
model at capacity                  503    True  upstream    not 429 — that is our quota on them
model returned inva

## 10 · Questions for the ADR

1. **Which endpoints stream, and what stops somebody adding streaming to one
   that returns structured output?** A rule nobody can point at is a rule that
   gets broken by the next person in a hurry.

2. **For every 4xx you return, can the caller actually fix it?** If not, it is
   probably a 5xx and you are blaming them for your problem.

3. **Does a developer on the consuming team branch on `retryable`?** Not
   "is it documented" — have you asked, and what did they say?

4. **What is your bad-request rate, and what does it cost you today?** If you
   do not know, you cannot argue for validation at the edge.

5. **A replayed idempotency key returns the original 202.** What returns a 409,
   and is that difference in the OpenAPI description or only in your head?

6. **What is your review trigger** for re-testing this surface?

---

### The five scenarios, in one line each

| | | |
|---|---|---|
| **1** | Unexpected field | Reject at the **gateway** — cheap, and a data protection control |
| **2** | Unreadable document | **422** — the caller can fix it |
| **3** | Invalid model JSON | **500** — the caller did not cause it, and it should page you |
| **4** | Client retries | **Idempotency key** — replay returns the original 202, not a 409 |
| **5** | Mid-stream failure | **You cannot return 422.** Validate before the first byte |

And the one that outranks them:

> An HTTP status is a category. `retryable` is an instruction.
> If nobody consumes it, you have designed documentation, not behaviour.

## 11 · ADR answers

**1. Which endpoints stream, and what stops somebody adding streaming to one that returns structured output?**
Only `GET /case-assessments/{id}/narrative` streams, and it streams prose to a human. `POST /case-application` returns `202` + ticket and never streams; the structured decision profile is only ever returned whole, after schema validation.
*Decision — enforce it, don't just state it:*
- Mark each operation in the OpenAPI spec with `x-streaming: true|false`; only the narrative endpoint may be `true`.
- A contract test in CI fails the build if any operation that returns the decision-profile schema declares `text/event-stream` or chunked output.
- A CODEOWNERS entry on the OpenAPI file, so a streaming change needs API-owner review.

**2. For every 4xx you return, can the caller actually fix it?**
Yes, checked against the Section 9 taxonomy:

| Status | Situation | Caller's fix |
|---|---|---|
| 400 | unexpected field | remove the field |
| 400 | missing idempotency key | add the `Idempotency-Key` header |
| 409 | key reused with a different body | use a new key, or resend the original body |
| 422 | document unreadable | supply a clearer scan or a text-based PDF |
| 422 | content filter blocked | change the content; the detail names the category |

Everything the caller cannot fix is a 5xx (500 invalid model output, 503 model capacity, 504 extraction timeout). `429` is reserved for the caller's own quota. The taxonomy passes this test.

**3. Does a developer on the consuming team branch on `retryable`?**
*Status: unconfirmed. Section 8 is a simulation, not evidence about their client.*
Position to take: the field is only worth shipping if it is read. Before go-live we send the consuming team the written question from Section 8 and record their answer here. Go-live is gated on it:
- If they read `retryable` -> done.
- If they retry on "5xx" -> pick one deliberately: (a) get them to read the field, (b) remove the retryable 5xx cases (move to a scheme where the heuristic works), or (c) accept the waste. Section 8 puts (c) at about 78 wasted calls and 35 lost cases per 1,000 requests, against 0 and 0 for a client that reads the field.
- No answer -> treat the client as "status only" and plan for that cost.

**4. What is your bad-request rate, and what does it cost you today?**
*Status: not measured. `ERROR_MIX` is illustrative, not production data.*
What the notebook does establish is the unit cost: a request rejected at the gateway costs about 2 ms and $0.00; the same request caught inside the function costs about 11.7 s and $0.014 (Document Intelligence + generation), and may be wrongly accepted. At that ratio, every 1,000 bad requests rejected late cost about $14 and about 3 hours of compute, and the edge check is what stops unwanted fields such as `applicant_ssn` reaching storage and logs.
Action: after two weeks of traffic, report the gateway 400 rate, the share of 4xx per type, and the cost avoided as `bad_requests x $0.014`. Until then the case for edge validation rests on the data-protection argument, not on a measured saving.

**5. A replayed idempotency key returns the original 202. What returns a 409, and is that difference documented?**
- Same key + same body -> `202` with the original `assessment_id` (no new work).
- Same key + different body -> `409 idempotency-key-reused`, `retryable: false`.
- No key -> `400 missing-idempotency-key`.

This is in the code and in `api_taxonomy.json`. *Decision:* publish it in the OpenAPI description for `POST /case-application` (the three outcomes above, with the `409` response and example body) so a consuming developer sees it without reading our code. Whether the shared spec already says this is unverified and should be checked against the live spec.

**6. What is your review trigger for re-testing this surface?**
Re-run the five scenarios and the taxonomy export when any of these happens:
- the request schema or the error taxonomy changes (a new error type, or a changed status or `retryable` value);
- the model, deployment or `API_VERSION` changes (this is what shifts invalid-JSON and content-filter behaviour);
- the consuming team's client or retry logic changes;
- a production signal: invalid-model-output rate rising, or gateway 400 rate moving noticeably from its baseline;
- otherwise, a fixed quarterly review.
Owner: the API owner for DecisionStream.

---

**Open items that need outside input:** #3 (the consuming team's answer) and #4 (production logs). Everything else above is a decision made here.